# Integrated Experiment Runner

This notebook consolidates the major experiments used in the Adaptive Swarm benchmarking project.

The goal is to avoid maintaining many fragmented notebooks. Dataset-level differences, such as universe, time horizon, and pre/post-normalisation settings, are handled through configuration. Label EDA, feature-set selection, model fitting, ranking evaluation, subgroup diagnostics, temporal diagnostics, and Excel exports are handled by one consistent experiment pipeline.

**Main design principles**

1. Preserve EDA for each label before modelling.
2. Reuse shared modelling and metric functions whenever possible.
3. Keep variable names consistent across the full notebook.
4. Explain each section before the code.
5. Export every major result table to Excel.
6. End with a checklist of tasks completed by this notebook.

## 1. Imports, global settings, and output folders

This section imports the required libraries and defines global variables used throughout the notebook. Keeping column names, paths, and random seeds in one place reduces variable inconsistency across later sections.

In [ ]:
import os
import json
import math
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier,
)
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    matthews_corrcoef,
    cohen_kappa_score,
    log_loss,
    confusion_matrix,
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS = -1

# Canonical column names used across the notebook.
INDEX_COL = 'IndexReference'
DATE_COL = 'attr__timestamp'
TICKER_COL = 'attr__ticker'
SIC2_COL = 'attr__sic2'
SIC_COL = 'attr__sic_code'
SIC_DESC_COL = 'attr__sic_description'
YEAR_COL = 'year'
SPLIT_COL = 'split'

TRUE_COL = 'y_true'
PRED_COL = 'y_pred'
SCORE_COL = 'prediction_score'
SIGNAL_SCORE_COL = 'signal_score'
DIRECTION_COL = 'direction'
CONFIDENCE_COL = 'confidence'

RL_TRAINABLE_COL = 'label__rl.trainable'

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'integrated_experiment_outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
MODEL_DIR = OUTPUT_DIR / 'models'
for d in [OUTPUT_DIR, TABLE_DIR, PREDICTION_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print('Output directory:', OUTPUT_DIR.resolve())


### Modification note — imports and global variables

This cell now imports additional evaluation metrics used by the project requirements, including explained variance, Matthews correlation coefficient, Cohen's kappa, log loss, and confusion matrix. It also defines `INDEX_COL`, `signal_score`, `direction`, `confidence`, and `label__rl.trainable` constants so the same variable names are used consistently throughout the notebook.

## 2. Dataset configuration

This section defines where the train, validation, and test files are located. Only the dataset paths should need to change when switching between Universe, time horizon, or pre/post-normalisation versions. The rest of the notebook runs from the same `DATASET_CONFIG` structure.

In [ ]:
DATASET_CONFIG = {
    'dataset_name': 'universe_100_full_post_normal_0527',
    'train_path': 'PATH_TO_TRAIN_FILE.parquet',
    'valid_path': 'PATH_TO_VALID_FILE.parquet',
    'test_path':  'PATH_TO_TEST_FILE.parquet',
    'project_b_feature_file': 'feature_project_b.txt',
    'universe': 'universe_100',
    'time_horizon': 'full',
    'normalisation_setting': 'post_normal',
}

## 3. Data loading utilities

This section loads and prepares the data in a consistent format. It supports flattened parquet/csv files and nested jsonl/json files. The downstream EDA and modelling sections always receive the same structure: one DataFrame per split in the `frames` dictionary.

In [ ]:
def load_flatten_jsonl(path):
    """
    Load and flatten an Adaptive Swarm model-ready JSONL file.

    Supported record structures:
    1. Official model-ready format:
       {"section": "data", "data": {"IndexReference": ..., "Attributes": {}, "Features": {}, "Labels": {}}}
    2. Already-flattened or earlier lower-case format:
       {"IndexReference": ..., "attributes": {}, "features": {}, "labels": {}}

    Output convention:
    - Attributes -> attr__*
    - Features   -> feature__*
    - Labels     -> label__*
    - IndexReference is preserved as the primary row join key for downstream prediction submission.
    """
    rows = []
    path = Path(path)

    def _open_text_file(p):
        if p.suffix.lower() == '.gz':
            import gzip
            return gzip.open(p, 'rt', encoding='utf-8')
        return p.open('r', encoding='utf-8')

    with _open_text_file(path) as f:
        for line in f:
            if not line.strip():
                continue

            raw_record = json.loads(line)

            # Skip header rows in official model-ready JSONL files.
            if raw_record.get('section') == 'header':
                continue

            # Official structure stores the useful row inside raw_record['data'].
            if raw_record.get('section') == 'data':
                record = raw_record.get('data', {}) or {}
            else:
                record = raw_record

            if not isinstance(record, dict):
                continue

            attrs = (
                record.get('Attributes')
                or record.get('attributes')
                or record.get('attrs')
                or {}
            )
            feats = record.get('Features') or record.get('features') or {}
            labs = record.get('Labels') or record.get('labels') or {}

            row = {INDEX_COL: record.get('IndexReference', raw_record.get('IndexReference'))}

            for k, v in attrs.items():
                row[k if str(k).startswith('attr__') else f'attr__{k}'] = v

            for k, v in feats.items():
                row[k if str(k).startswith('feature__') else f'feature__{k}'] = v

            for k, v in labs.items():
                row[k if str(k).startswith('label__') else f'label__{k}'] = v

            # Preserve simple scalar fields from the row for auditability.
            for k, v in record.items():
                if (
                    k not in ['Attributes', 'attributes', 'attrs', 'Features', 'features', 'Labels', 'labels']
                    and not isinstance(v, (dict, list))
                ):
                    row.setdefault(k, v)

            rows.append(row)

    df = pd.DataFrame(rows)
    print(f'Flattened JSONL shape: {df.shape}')
    return df


def load_jsonl_zip(path):
    """
    Load a .jsonl.zip file where the archive contains one JSONL file.
    This matches the project documentation's recommended model-ready data format.
    """
    import zipfile
    path = Path(path)
    rows = []

    with zipfile.ZipFile(path, 'r') as zf:
        jsonl_names = [name for name in zf.namelist() if name.lower().endswith('.jsonl')]
        if not jsonl_names:
            raise ValueError(f'No .jsonl file found inside zip archive: {path}')

        with zf.open(jsonl_names[0], 'r') as f:
            for raw_line in f:
                line = raw_line.decode('utf-8')
                if not line.strip():
                    continue
                raw_record = json.loads(line)
                if raw_record.get('section') == 'header':
                    continue
                record = raw_record.get('data', {}) if raw_record.get('section') == 'data' else raw_record
                if not isinstance(record, dict):
                    continue

                attrs = record.get('Attributes') or record.get('attributes') or record.get('attrs') or {}
                feats = record.get('Features') or record.get('features') or {}
                labs = record.get('Labels') or record.get('labels') or {}

                row = {INDEX_COL: record.get('IndexReference', raw_record.get('IndexReference'))}
                for k, v in attrs.items():
                    row[k if str(k).startswith('attr__') else f'attr__{k}'] = v
                for k, v in feats.items():
                    row[k if str(k).startswith('feature__') else f'feature__{k}'] = v
                for k, v in labs.items():
                    row[k if str(k).startswith('label__') else f'label__{k}'] = v
                for k, v in record.items():
                    if (
                        k not in ['Attributes', 'attributes', 'attrs', 'Features', 'features', 'Labels', 'labels']
                        and not isinstance(v, (dict, list))
                    ):
                        row.setdefault(k, v)
                rows.append(row)

    df = pd.DataFrame(rows)
    print(f'Flattened zipped JSONL shape: {df.shape}')
    return df


def load_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    name = path.name.lower()
    suffix = path.suffix.lower()

    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.jsonl' or name.endswith('.jsonl.gz'):
        return load_flatten_jsonl(path)
    if name.endswith('.jsonl.zip'):
        return load_jsonl_zip(path)
    if suffix == '.json':
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)

    raise ValueError(f'Unsupported file format: {path}')


def standardise_frame(df, split_name):
    df = df.copy()
    df[SPLIT_COL] = split_name

    if INDEX_COL not in df.columns:
        # Fallback only for older flattened datasets without IndexReference.
        # Official simulator submission requires the source IndexReference, so this should be audited.
        df[INDEX_COL] = np.arange(len(df))
        print(f'Warning: {INDEX_COL} missing in {split_name}; generated sequential fallback index.')

    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
        df[YEAR_COL] = df[DATE_COL].dt.year

    for col in [TICKER_COL, SIC2_COL, SIC_COL, SIC_DESC_COL]:
        if col not in df.columns:
            df[col] = np.nan

    return df


def load_dataset_from_config(config):
    frames = {}
    for split_name in ['train', 'valid', 'test']:
        df = load_table(config[f'{split_name}_path'])
        frames[split_name] = standardise_frame(df, split_name)
        print(f'{split_name}: {frames[split_name].shape}')
    all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
    return frames, all_data

# frames, all_data = load_dataset_from_config(DATASET_CONFIG)


### Modification note — JSONL loading and IndexReference preservation

This cell now supports the official model-ready JSONL structure with `section: header` and `section: data`, plus `.jsonl.zip` files. `IndexReference` is preserved as the primary join key for downstream prediction submission and trade simulation.

## 4. Export manager

This section creates a central export system. Every EDA table, metric table, comparison table, diagnostic table, and checklist can be registered once and then exported automatically. This ensures every important DataFrame is available as Excel output.

In [ ]:
if 'EXPORTED_TABLES' not in globals():
    EXPORTED_TABLES = {}


def safe_table_name(name, max_len=120):
    name = re.sub(r'[^A-Za-z0-9_\-]+', '_', str(name))
    name = re.sub(r'_+', '_', name).strip('_')
    return name[:max_len] or 'table'


def safe_sheet_name(name):
    name = re.sub(r'[\[\]\:\*\?\/\\]', '_', str(name))[:31]
    return name or 'Sheet'


def safe_file_name(name, max_len=120):
    name = str(name)
    for ch in ['\\', '/', ':', '*', '?', '"', '<', '>', '|']:
        name = name.replace(ch, '_')
    return name[:max_len] or 'file'


def make_excel_safe(df):
    """
    Convert DataFrame values into Excel-safe formats.
    The main fix is removing timezone information from datetime columns before Excel export.
    """
    df = df.copy()

    for col in df.columns:
        if pd.api.types.is_datetime64tz_dtype(df[col]):
            df[col] = df[col].dt.tz_convert(None)
        elif df[col].dtype == 'object':
            df[col] = df[col].apply(
                lambda x: x.tz_convert(None)
                if isinstance(x, pd.Timestamp) and x.tzinfo is not None
                else x
            )

    return df


def register_table(name, df, export_immediately=False):
    if df is None:
        return None
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    key = safe_table_name(name)
    EXPORTED_TABLES[key] = df.copy()

    if export_immediately:
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        path = TABLE_DIR / f'{key}.xlsx'
        make_excel_safe(df).to_excel(path, index=False)
        print('Exported:', path)

    return df


def export_registered_tables(workbook_name=None, export_individual_files=True, max_sheet_rows=1_000_000):
    if workbook_name is None:
        workbook_name = f'integrated_experiment_tables_{RUN_TIMESTAMP}.xlsx'

    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    workbook_path = TABLE_DIR / workbook_name

    used = set()
    n_written = 0
    skipped_rows = []

    with pd.ExcelWriter(workbook_path, engine='openpyxl') as writer:
        for name, df in EXPORTED_TABLES.items():
            if df is None:
                skipped_rows.append({'table_name': name, 'reason': 'df_is_none'})
                continue
            if not isinstance(df, pd.DataFrame):
                skipped_rows.append({'table_name': name, 'reason': f'not_dataframe_{type(df)}'})
                continue
            if len(df) == 0:
                skipped_rows.append({'table_name': name, 'reason': 'empty_dataframe'})
                continue

            sheet = safe_sheet_name(name)
            base = sheet
            i = 1
            while sheet in used:
                suffix = f'_{i}'
                sheet = safe_sheet_name(base[:31 - len(suffix)] + suffix)
                i += 1
            used.add(sheet)

            excel_df = make_excel_safe(df)
            excel_df.head(max_sheet_rows).to_excel(writer, sheet_name=sheet, index=False)
            n_written += 1

        # Excel workbooks must contain at least one visible sheet.
        if n_written == 0:
            readme = pd.DataFrame([
                {
                    'message': 'No non-empty registered tables were available for export.',
                    'possible_reason_1': 'EXPORTED_TABLES is empty.',
                    'possible_reason_2': 'The workflow cells that call register_table() have not been run.',
                    'possible_reason_3': 'All registered tables were empty.',
                    'next_step': 'Run EDA/model/diagnostic cells, then export again.',
                }
            ])
            readme.to_excel(writer, sheet_name='README', index=False)

    if export_individual_files:
        for name, df in EXPORTED_TABLES.items():
            if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
                continue
            excel_df = make_excel_safe(df)
            excel_df.head(max_sheet_rows).to_excel(TABLE_DIR / f'{safe_file_name(name)}.xlsx', index=False)

    print('Integrated workbook exported to:', workbook_path.resolve())
    print('Number of registered tables:', len(EXPORTED_TABLES))
    print('Number of non-empty tables written:', n_written)

    if skipped_rows:
        display(pd.DataFrame(skipped_rows))

    return workbook_path


### Modification note — robust export manager

This cell no longer clears `EXPORTED_TABLES` when rerun. It also removes timezone information before Excel export, protects against empty workbooks, and uses safe file names for individual table exports.

## 5. Label construction and target configuration

This section standardises all target definitions. It preserves the EDA for each label and makes target handling explicit, so later modelling functions do not rely on hidden variable names.

Supported target families include perfect-hindsight labels, RL expert action labels, RL long-side quality/reward labels, and current PnL labels.

In [ ]:
def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def make_pi_hindsight_entry_long_6bins(series):
    s = pd.to_numeric(series, errors='coerce')
    out = pd.Series(np.nan, index=s.index)
    out[s == 0] = 0
    out[(s > 0) & (s < 0.1)] = 1
    out[(s >= 0.1) & (s < 0.2)] = 2
    out[(s >= 0.2) & (s < 0.3)] = 3
    out[(s >= 0.3) & (s < 0.4)] = 4
    out[s >= 0.4] = 5
    return out.astype('Int64')


def add_derived_targets(df):
    df = df.copy()

    pi_col = first_existing_column(df, ['label__pi_hindsight_entry_long', 'label__pi_long_entry', 'pi_hindsight_entry_long'])
    if pi_col is not None:
        df['target__pi_hindsight_entry_long'] = pd.to_numeric(df[pi_col], errors='coerce')
        df['target__pi_hindsight_entry_positive'] = (df['target__pi_hindsight_entry_long'] > 0).astype('Int64')
        df['target__pi_hindsight_entry_original'] = (df['target__pi_hindsight_entry_long'] >= 0.4).astype('Int64')
        df['target__pi_hindsight_entry_6bins'] = make_pi_hindsight_entry_long_6bins(df['target__pi_hindsight_entry_long'])

    col = first_existing_column(df, ['label__rl.expert_action', 'label__rl_expert_action'])
    if col is not None:
        df['target__rl_expert_action'] = df[col]

    col = first_existing_column(df, ['label__rl.long_is_best', 'label__rl_long_is_best'])
    if col is not None:
        df['target__rl_long_is_best'] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    col = first_existing_column(df, ['label__rl.long.action_quality', 'label__rl_long_action_quality', 'label__rl.long.action_label'])
    if col is not None:
        df['target__rl_long_action_quality'] = pd.to_numeric(df[col], errors='coerce')

    col = first_existing_column(df, ['label__rl.reward.long', 'label__rl_reward_long', 'label__rl.long.reward'])
    if col is not None:
        df['target__rl_long_reward'] = pd.to_numeric(df[col], errors='coerce')

    col = first_existing_column(df, ['label__rl.long.current_pnl', 'label__rl_long_current_pnl', 'label__rl.long_current_pnl', 'label__current_pnl'])
    if col is not None:
        df['target__rl_long_current_pnl'] = pd.to_numeric(df[col], errors='coerce')

    reward_long = first_existing_column(df, ['label__rl.reward.long', 'target__rl_long_reward'])
    reward_short = first_existing_column(df, ['label__rl.reward.short', 'label__rl_reward_short'])
    reward_no_trade = first_existing_column(df, ['label__rl.reward.no_trade', 'label__rl_reward_no_trade', 'label__rl.reward.hold'])

    if reward_long and reward_short and reward_no_trade:
        r_long = pd.to_numeric(df[reward_long], errors='coerce')
        r_short = pd.to_numeric(df[reward_short], errors='coerce')
        r_no_trade = pd.to_numeric(df[reward_no_trade], errors='coerce')

        if 'target__rl_long_is_best' not in df.columns:
            df['target__rl_long_is_best'] = ((r_long > r_short) & (r_long > r_no_trade)).astype('Int64')

        best_non_long_reward = pd.concat([r_short, r_no_trade], axis=1).max(axis=1)
        df['target__rl_long_reward_margin'] = r_long - best_non_long_reward
        df['target__rl_long_high_confidence'] = (df['target__rl_long_reward_margin'] >= 0.05).astype('Int64')

    return df


TARGET_CONFIGS = {
    'pi_hindsight_entry_long': {'column': 'target__pi_hindsight_entry_long', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous perfect-hindsight long-entry score.'},
    'pi_hindsight_entry_positive': {'column': 'target__pi_hindsight_entry_positive', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary target: pi score > 0.'},
    'pi_hindsight_entry_original': {'column': 'target__pi_hindsight_entry_original', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary target: pi score >= 0.4.'},
    'pi_hindsight_entry_6bins': {'column': 'target__pi_hindsight_entry_6bins', 'task': 'multiclass', 'direction': 'higher_is_better', 'description': 'Six-bin ordinal perfect-hindsight target.'},
    'rl_expert_action': {'column': 'target__rl_expert_action', 'task': 'multiclass', 'direction': 'action', 'description': 'RL evaluator expert action classification.'},
    'rl_long_is_best': {'column': 'target__rl_long_is_best', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary long-side target: long is best action.'},
    'rl_long_high_confidence': {'column': 'target__rl_long_high_confidence', 'task': 'binary', 'positive_label': 1, 'direction': 'higher_is_better', 'description': 'Binary target: long reward exceeds the best non-long alternative by a margin.'},
    'rl_long_action_quality': {'column': 'target__rl_long_action_quality', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous long action quality.'},
    'rl_long_reward': {'column': 'target__rl_long_reward', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous long reward.'},
    'rl_long_reward_margin': {'column': 'target__rl_long_reward_margin', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Long reward minus the best non-long reward.'},
    'rl_long_current_pnl': {'column': 'target__rl_long_current_pnl', 'task': 'regression', 'direction': 'higher_is_better', 'description': 'Continuous long current PnL.'},
}


def apply_target_construction(frames):
    updated = {split: add_derived_targets(df) for split, df in frames.items()}
    all_data = pd.concat(updated.values(), ignore_index=True, sort=False)
    return updated, all_data

# frames, all_data = apply_target_construction(frames)


### Modification note — target construction

This cell now includes derived RL reward-margin and high-confidence long targets when the long, short, and no-trade reward labels are available. The target registry was updated so these derived targets can be selected consistently by the experiment runner.

## 6. Label EDA

This section preserves EDA for each label before model training. This is important because the modelling results can only be interpreted correctly after checking label sparsity, split stability, time variation, ticker heterogeneity, and sector heterogeneity. All EDA tables are registered for Excel export.

In [ ]:
def available_target_configs(df, target_configs=TARGET_CONFIGS):
    return {name: cfg for name, cfg in target_configs.items() if cfg['column'] in df.columns}


def label_overall_summary(all_data, target_configs=TARGET_CONFIGS):
    rows = []
    for target_name, cfg in available_target_configs(all_data, target_configs).items():
        col = cfg['column']
        y = all_data[col]
        row = {'target_name': target_name, 'column': col, 'task': cfg['task'], 'description': cfg.get('description', ''), 'n_rows': len(y), 'n_non_missing': int(y.notna().sum()), 'missing_rate': y.isna().mean(), 'n_unique': y.nunique(dropna=True)}
        if cfg['task'] == 'regression':
            yy = pd.to_numeric(y, errors='coerce')
            row.update({'mean': yy.mean(), 'std': yy.std(), 'min': yy.min(), 'p01': yy.quantile(0.01), 'p05': yy.quantile(0.05), 'median': yy.median(), 'p95': yy.quantile(0.95), 'p99': yy.quantile(0.99), 'max': yy.max(), 'zero_rate': (yy == 0).mean(), 'positive_rate': (yy > 0).mean(), 'negative_rate': (yy < 0).mean()})
        else:
            if cfg['task'] == 'binary':
                row['positive_rate'] = (y == cfg.get('positive_label', 1)).mean()
            counts = y.value_counts(dropna=False, normalize=True)
            for k, v in counts.head(20).items():
                row[f'class_rate_{k}'] = v
        rows.append(row)
    return register_table('label_eda_overall_summary', pd.DataFrame(rows))


def label_distribution_by_split(all_data, target_configs=TARGET_CONFIGS):
    rows = []
    for target_name, cfg in available_target_configs(all_data, target_configs).items():
        col = cfg['column']
        for split_name, g in all_data.groupby(SPLIT_COL, dropna=False):
            y = g[col]
            base = {'target_name': target_name, 'split': split_name, 'task': cfg['task'], 'n_rows': len(g), 'n_non_missing': int(y.notna().sum()), 'missing_rate': y.isna().mean()}
            if cfg['task'] == 'regression':
                yy = pd.to_numeric(y, errors='coerce')
                base.update({'mean': yy.mean(), 'std': yy.std(), 'median': yy.median(), 'zero_rate': (yy == 0).mean(), 'positive_rate': (yy > 0).mean(), 'negative_rate': (yy < 0).mean()})
                rows.append(base)
            else:
                for klass, rate in y.value_counts(dropna=False, normalize=True).items():
                    row = base.copy(); row['class'] = klass; row['class_rate'] = rate; row['class_count'] = int((y == klass).sum()) if pd.notna(klass) else int(y.isna().sum()); rows.append(row)
    return register_table('label_eda_by_split', pd.DataFrame(rows))


def label_distribution_by_group(all_data, group_col, target_configs=TARGET_CONFIGS, min_rows=30):
    if group_col not in all_data.columns:
        return register_table(f'label_eda_by_{group_col}', pd.DataFrame())
    rows = []
    for target_name, cfg in available_target_configs(all_data, target_configs).items():
        col = cfg['column']
        for group_value, g in all_data.groupby(group_col, dropna=False):
            if len(g) < min_rows:
                continue
            y = g[col]
            row = {'target_name': target_name, 'group_col': group_col, 'group_value': group_value, 'task': cfg['task'], 'n_rows': len(g), 'n_non_missing': int(y.notna().sum()), 'missing_rate': y.isna().mean()}
            if cfg['task'] == 'regression':
                yy = pd.to_numeric(y, errors='coerce')
                row.update({'mean': yy.mean(), 'std': yy.std(), 'median': yy.median(), 'zero_rate': (yy == 0).mean(), 'positive_rate': (yy > 0).mean(), 'negative_rate': (yy < 0).mean()})
            else:
                if cfg['task'] == 'binary':
                    row['positive_rate'] = (y == cfg.get('positive_label', 1)).mean()
                row['mode'] = y.mode(dropna=True).iloc[0] if y.notna().any() else np.nan
                row['mode_rate'] = y.value_counts(normalize=True, dropna=True).iloc[0] if y.notna().any() else np.nan
            rows.append(row)
    return register_table(f'label_eda_by_{safe_table_name(group_col)}', pd.DataFrame(rows))


def run_all_label_eda(all_data):
    return {
        'overall': label_overall_summary(all_data),
        'by_split': label_distribution_by_split(all_data),
        'by_year': label_distribution_by_group(all_data, YEAR_COL),
        'by_ticker': label_distribution_by_group(all_data, TICKER_COL),
        'by_sic2': label_distribution_by_group(all_data, SIC2_COL),
    }

# label_eda_tables = run_all_label_eda(all_data)

## 7. Feature-set construction

This section defines reusable feature sets. The feature-set logic is separated from the modelling code so that different targets can be tested against the same feature families.

In [ ]:
EXCLUDE_COLUMNS = {SPLIT_COL, YEAR_COL, DATE_COL, TICKER_COL, SIC2_COL, SIC_COL, SIC_DESC_COL}


def read_feature_file(feature_file):
    if feature_file is None:
        return []
    path = Path(feature_file)
    if not path.exists():
        print(f'Feature file not found: {path}. Falling back to automatic numeric features.')
        return []
    raw = [line.strip() for line in path.open('r', encoding='utf-8') if line.strip() and not line.strip().startswith('#')]
    return [col if col.startswith('feature__') else f'feature__{col}' for col in raw]


def get_numeric_feature_candidates(df):
    numeric_cols = df.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    out = []
    for col in numeric_cols:
        if col in EXCLUDE_COLUMNS:
            continue
        if col.startswith('label__') or col.startswith('target__') or col.startswith('attr__'):
            continue
        if col.startswith('feature__'):
            out.append(col)
    return sorted(set(out))


def select_features_by_keywords(all_features, include_keywords=None, exclude_keywords=None):
    include_keywords = include_keywords or []
    exclude_keywords = exclude_keywords or []
    out = []
    for col in all_features:
        c = col.lower()
        include_ok = True if not include_keywords else any(k.lower() in c for k in include_keywords)
        exclude_ok = not any(k.lower() in c for k in exclude_keywords)
        if include_ok and exclude_ok:
            out.append(col)
    return sorted(set(out))


def build_feature_sets(all_data, config=DATASET_CONFIG):
    auto_numeric = get_numeric_feature_candidates(all_data)
    project_b_raw = read_feature_file(config.get('project_b_feature_file'))
    project_b = [c for c in project_b_raw if c in all_data.columns]
    if len(project_b) == 0:
        project_b = auto_numeric.copy()

    feature_sets = {
        'combined_project_b': project_b,
        'combined_all_numeric_features': auto_numeric,
        'fundamentals_only': select_features_by_keywords(auto_numeric, ['fund', 'asset', 'liabil', 'equity', 'cash', 'debt', 'revenue', 'income', 'earn', 'profit', 'margin', 'eps', 'book', 'balance', 'report', 'quarter', 'ttm', 'filing']),
        'daily_valuation_only': select_features_by_keywords(auto_numeric, ['pe', 'pb', 'ps', 'ev', 'valuation', 'market_cap', 'price_to', 'yield', 'dividend', 'multiple']),
        'momentum_volatility_only': select_features_by_keywords(auto_numeric, ['return', 'ret', 'momentum', 'mom', 'vol', 'volatility', 'atr', 'rsi', 'macd', 'sma', 'ema', 'drawdown', 'trend', 'beta']),
        'macro_regime_only': select_features_by_keywords(auto_numeric, ['macro', 'regime', 'inflation', 'vix', 'yield_curve', 'rate', 'treasury', 'credit', 'index', 'sector', 'market']),
        'algorithmic_signals_only': select_features_by_keywords(auto_numeric, ['signal', 'alpha', 'score', 'rank', 'swarm', 'algo', 'model', 'entry', 'exit', 'confidence'], ['label', 'target']),
    }
    feature_sets = {k: v for k, v in feature_sets.items() if len(v) > 0}
    summary = pd.DataFrame([{'feature_set': k, 'n_features': len(v)} for k, v in feature_sets.items()])
    register_table('feature_set_summary', summary)
    return feature_sets

# FEATURE_SET_CONFIGS = build_feature_sets(all_data, DATASET_CONFIG)

## 8. Shared modelling utilities

This section contains functions that can be shared across regression, binary classification, and multiclass classification. Task-specific model dictionaries are separated where the analysis differs.

In [ ]:
def make_numeric_preprocessor(scale=False):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    return Pipeline(steps)


def make_regression_models():
    models = {
        'DummyMean': DummyRegressor(strategy='mean'),
        'Ridge': make_pipeline(make_numeric_preprocessor(True), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        'Lasso': make_pipeline(make_numeric_preprocessor(True), Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=5000)),
        'ElasticNet': make_pipeline(make_numeric_preprocessor(True), ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMRegressor
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def make_binary_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'DummyStratified': DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
        'Logistic': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def make_multiclass_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'LogisticMultinomial': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=3000, multi_class='auto', random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def get_models_for_task(task):
    if task == 'regression':
        return make_regression_models()
    if task == 'binary':
        return make_binary_models()
    if task == 'multiclass':
        return make_multiclass_models()
    raise ValueError(f'Unknown task: {task}')


def prepare_xy(df, feature_cols, target_col, task):
    cols = [c for c in feature_cols if c in df.columns] + [target_col]
    data = df[cols].copy().dropna(subset=[target_col])
    X = data[[c for c in feature_cols if c in data.columns]]
    y = data[target_col]
    if task == 'regression':
        y = pd.to_numeric(y, errors='coerce')
        valid = y.notna()
        X = X.loc[valid]
        y = y.loc[valid]
    else:
        valid = y.notna()
        X = X.loc[valid]
        y = y.loc[valid]
    return X, y


def is_rl_target(target_name):
    return str(target_name).startswith('rl_')


def filter_rows_for_target(df, target_name):
    """
    Apply target-specific row filtering.
    RL-derived targets use label__rl.trainable == 1 when that flag exists.
    Non-RL targets are left unchanged.
    """
    df = df.copy()
    if is_rl_target(target_name) and RL_TRAINABLE_COL in df.columns:
        return df[df[RL_TRAINABLE_COL] == 1].copy()
    return df


def safe_spearman(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='spearman')


def safe_pearson(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='pearson')


def get_positive_proba(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        classes = getattr(model, 'classes_', None)
        if classes is None and hasattr(model, 'named_steps'):
            classes = getattr(list(model.named_steps.values())[-1], 'classes_', None)
        if classes is not None:
            classes = list(classes)
            if 1 in classes:
                return proba[:, classes.index(1)]
            if True in classes:
                return proba[:, classes.index(True)]
        return proba[:, -1]
    if hasattr(model, 'decision_function'):
        score = model.decision_function(X)
        return 1 / (1 + np.exp(-score))
    return None


def build_prediction_frame(df, y_true, y_pred, score, target_name, task, feature_set_name, model_name, split_name):
    """
    Build a row-level prediction DataFrame.
    IndexReference is preserved because it is the official join key for downstream simulation.
    """
    meta_cols = [INDEX_COL, DATE_COL, YEAR_COL, TICKER_COL, SIC2_COL]
    meta = df.loc[y_true.index, [c for c in meta_cols if c in df.columns]].copy()
    out = meta.copy()
    out['target_name'] = target_name
    out['task'] = task
    out['feature_set'] = feature_set_name
    out['model'] = model_name
    out[SPLIT_COL] = split_name
    out[TRUE_COL] = np.asarray(y_true)
    out[PRED_COL] = np.asarray(y_pred)
    out[SCORE_COL] = np.asarray(score) if score is not None else np.asarray(y_pred)
    return out


def add_signal_interface(predictions_df, target_configs=TARGET_CONFIGS):
    """
    Convert raw model outputs into the common prediction interface:
    signal_score, direction, and confidence.

    signal_score is rank-normalised within each target-feature-model-split group so that
    outputs from different model types are comparable for downstream ranking.
    """
    if predictions_df is None or len(predictions_df) == 0:
        return predictions_df

    df = predictions_df.copy()
    group_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    parts = []

    for _, g in df.groupby(group_cols, dropna=False):
        g = g.copy()
        target_name = g['target_name'].iloc[0]
        target_cfg = target_configs.get(target_name, {})
        direction_type = target_cfg.get('direction', 'higher_is_better')

        raw_score = pd.to_numeric(g[SCORE_COL], errors='coerce')
        rank_pct = raw_score.rank(method='average', pct=True)
        signal_score = 1 - rank_pct if direction_type == 'lower_is_better' else rank_pct
        signal_score = signal_score.clip(0, 1)

        g[SIGNAL_SCORE_COL] = signal_score
        g[CONFIDENCE_COL] = ((signal_score - 0.5).abs() * 2).clip(0.01, 1.0)

        # Default direction mapping: this notebook mainly produces long-side benchmark signals.
        g[DIRECTION_COL] = np.where(signal_score >= 0.5, 'long', 'no_trade')

        # Optional configurable mapping for action-style targets.
        if direction_type == 'action':
            action_map = {0: 'no_trade', 1: 'long', 2: 'short'}
            numeric_pred = pd.to_numeric(g[PRED_COL], errors='coerce')
            mapped = numeric_pred.map(action_map)
            g[DIRECTION_COL] = mapped.fillna(g[DIRECTION_COL])

        parts.append(g)

    return pd.concat(parts, ignore_index=True, sort=False)


### Modification note — modelling utilities and prediction frame

This cell now adds Ridge and Lasso regression models, target-specific RL trainable filtering, `IndexReference` preservation in `build_prediction_frame`, and conversion of raw model scores into `signal_score`, `direction`, and `confidence`. Only `SIC2` is kept as the sector-level grouping field in prediction metadata.

## 9. Metric functions

This section separates task-specific metrics from shared modelling logic. Regression, binary classification, multiclass classification, calibration, and ranking metrics answer different research questions. Ranking and Top-K metrics are especially important because the practical question is whether the model ranks better long opportunities above weaker ones.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}

    yt = y_true.loc[valid]
    yp = y_pred.loc[valid]
    errors = yp - yt
    abs_errors = errors.abs()
    denom = yt.abs().sum()

    directional_accuracy = np.nan
    if yt.nunique() > 1 and yp.nunique() > 1:
        directional_accuracy = (np.sign(yt) == np.sign(yp)).mean()

    return {
        'n': int(valid.sum()),
        'mae': mean_absolute_error(yt, yp),
        'mse': mean_squared_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'explained_variance': explained_variance_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'pearson': safe_pearson(yt, yp),
        'spearman': safe_spearman(yt, yp),
        'information_coefficient': safe_pearson(yt, yp),
        'rank_information_coefficient': safe_spearman(yt, yp),
        'directional_accuracy': directional_accuracy,
        'weighted_mape': abs_errors.sum() / denom if denom > 0 else np.nan,
        'error_p50': abs_errors.quantile(0.50),
        'error_p90': abs_errors.quantile(0.90),
        'error_p95': abs_errors.quantile(0.95),
        'error_p99': abs_errors.quantile(0.99),
        'mean_y_true': yt.mean(),
        'mean_y_pred': yp.mean(),
        'std_y_true': yt.std(),
        'std_y_pred': yp.std(),
    }


def binary_metrics(y_true, y_pred, y_score=None):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}

    yt = y_true.loc[valid].astype(int)
    yp = y_pred.loc[valid].astype(int)

    try:
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    except Exception:
        tn = fp = fn = tp = np.nan

    out = {
        'n': int(valid.sum()),
        'accuracy': accuracy_score(yt, yp),
        'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'precision': precision_score(yt, yp, zero_division=0),
        'recall': recall_score(yt, yp, zero_division=0),
        'f1': f1_score(yt, yp, zero_division=0),
        'matthews_corrcoef': matthews_corrcoef(yt, yp) if yt.nunique() > 1 else np.nan,
        'cohen_kappa': cohen_kappa_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'positive_rate_true': yt.mean(),
        'positive_rate_pred': yp.mean(),
    }

    if y_score is not None:
        ys = pd.Series(y_score, index=yt.index).astype(float)
        out['mean_predicted_probability'] = ys.mean()
        if yt.nunique() > 1 and ys.nunique() > 1:
            ys_prob = np.clip(ys, 1e-6, 1 - 1e-6)
            out.update({
                'roc_auc': roc_auc_score(yt, ys),
                'pr_auc': average_precision_score(yt, ys),
                'brier_score': brier_score_loss(yt, np.clip(ys, 0, 1)),
                'log_loss': log_loss(yt, ys_prob),
                'spearman': safe_spearman(yt, ys),
            })

    return out


def multiclass_metrics(y_true, y_pred, y_proba=None):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}

    yt = y_true.loc[valid]
    yp = y_pred.loc[valid]
    out = {
        'n': int(valid.sum()),
        'accuracy': accuracy_score(yt, yp),
        'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'macro_f1': f1_score(yt, yp, average='macro', zero_division=0),
        'weighted_f1': f1_score(yt, yp, average='weighted', zero_division=0),
        'n_classes_true': yt.nunique(),
        'n_classes_pred': yp.nunique(),
    }

    try:
        out['spearman_class_rank'] = safe_spearman(pd.to_numeric(yt), pd.to_numeric(yp))
    except Exception:
        out['spearman_class_rank'] = np.nan

    return out


def calibration_table(y_true, y_score, n_bins=10):
    df = pd.DataFrame({TRUE_COL: pd.Series(y_true).astype(float), SCORE_COL: pd.Series(y_score).astype(float)}).dropna()
    if len(df) == 0:
        return pd.DataFrame()
    df[SCORE_COL] = df[SCORE_COL].clip(0, 1)
    df['prob_bin'] = pd.cut(df[SCORE_COL], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
    out = df.groupby('prob_bin', observed=False).agg(
        n=(TRUE_COL, 'size'),
        mean_predicted_probability=(SCORE_COL, 'mean'),
        true_positive_rate=(TRUE_COL, 'mean'),
    ).reset_index()
    out['calibration_error'] = out['mean_predicted_probability'] - out['true_positive_rate']
    return out


def daily_cross_sectional_spearman(pred_df, min_daily_rows=5):
    rows = []
    if DATE_COL not in pred_df.columns:
        return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        if len(g) < min_daily_rows:
            continue
        rows.append({
            DATE_COL: date,
            'n': len(g),
            'daily_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(),
        })
    return pd.DataFrame(rows)


def topk_diagnostics(pred_df, top_pct=0.10, min_daily_rows=10):
    rows = []
    if DATE_COL not in pred_df.columns:
        return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        g = g.dropna(subset=[TRUE_COL, SCORE_COL]).copy()
        if len(g) < min_daily_rows:
            continue
        k = max(1, int(math.ceil(len(g) * top_pct)))
        top = g.nlargest(k, SCORE_COL)
        bottom = g.nsmallest(k, SCORE_COL)
        y = pd.to_numeric(g[TRUE_COL], errors='coerce')
        top_y = pd.to_numeric(top[TRUE_COL], errors='coerce')
        bottom_y = pd.to_numeric(bottom[TRUE_COL], errors='coerce')
        row = {
            DATE_COL: date,
            'n': len(g),
            'k': k,
            'top_pct': top_pct,
            'overall_mean_true': y.mean(),
            'top_mean_true': top_y.mean(),
            'bottom_mean_true': bottom_y.mean(),
            'top_minus_bottom_spread': top_y.mean() - bottom_y.mean(),
        }
        unique_values = pd.Series(g[TRUE_COL]).dropna().unique()
        if set(unique_values).issubset({0, 1, False, True}):
            base_rate = y.mean()
            precision_at_k = top_y.mean()
            row.update({
                'base_positive_rate': base_rate,
                'precision_at_k': precision_at_k,
                'bottom_positive_rate': bottom_y.mean(),
                'lift_at_k': precision_at_k / base_rate if base_rate and base_rate > 0 else np.nan,
            })
        rows.append(row)
    return pd.DataFrame(rows)


def within_ticker_ranking(pred_df, min_rows=20):
    if TICKER_COL not in pred_df.columns:
        return pd.DataFrame()
    rows = []
    for ticker, g in pred_df.groupby(TICKER_COL, dropna=False):
        if len(g) < min_rows:
            continue
        rows.append({
            'ticker': ticker,
            'n': len(g),
            'within_ticker_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(),
        })
    return pd.DataFrame(rows)


def subgroup_prediction_metrics(pred_df, group_col, min_rows=30):
    if group_col not in pred_df.columns:
        return pd.DataFrame()
    rows = []
    for group_value, g in pred_df.groupby(group_col, dropna=False):
        if len(g) < min_rows:
            continue
        row = {
            'group_col': group_col,
            'group_value': group_value,
            'n': len(g),
            'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(),
            'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(),
            'spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]),
        }
        unique_values = pd.Series(g[TRUE_COL]).dropna().unique()
        try:
            if set(unique_values).issubset({0, 1, False, True}):
                row.update(binary_metrics(g[TRUE_COL], g[PRED_COL], g[SCORE_COL]))
            else:
                row.update(regression_metrics(g[TRUE_COL], g[SCORE_COL]))
        except Exception:
            pass
        rows.append(row)
    return pd.DataFrame(rows)


### Modification note — expanded metrics

This cell now adds the additional metrics requested for reporting and downstream diagnostics: MSE, explained variance, information coefficient, rank information coefficient, directional accuracy, weighted MAPE, error percentiles, MCC, Cohen's kappa, log loss, and confusion-matrix cells.

## 10. Main experiment runner

This section is the core integrated runner. It loops over target definitions, feature sets, and models while keeping the same variable names and output format. The runner automatically skips invalid combinations, such as unavailable target columns, empty feature sets, or targets with too few training examples.

In [ ]:
EXPERIMENT_CONFIG = {
    'target_names': None,
    'feature_set_names': None,
    'model_names': None,
    'min_train_rows': 100,
    'min_valid_or_test_rows': 50,
    'save_row_level_predictions': True,
    'run_calibration': True,
    'run_daily_ranking': True,
    'run_topk': True,
    'run_within_ticker_ranking': True,
    'run_subgroup_diagnostics': True,
    'topk_percentages': [0.05, 0.10],
}


def should_run_name(name, selected_names):
    return selected_names is None or name in selected_names


def fit_predict_single_experiment(frames, target_name, target_cfg, feature_set_name, feature_cols, model_name, model, config=EXPERIMENT_CONFIG):
    target_col = target_cfg['column']
    task = target_cfg['task']

    train_df = filter_rows_for_target(frames['train'], target_name)
    valid_df = filter_rows_for_target(frames['valid'], target_name)
    test_df = filter_rows_for_target(frames['test'], target_name)

    X_train, y_train = prepare_xy(train_df, feature_cols, target_col, task)

    if len(y_train) < config['min_train_rows']:
        return [], [], {'status': 'skipped', 'reason': 'too_few_train_rows', 'n_train': len(y_train)}
    if task in ['binary', 'multiclass'] and y_train.nunique(dropna=True) < 2:
        return [], [], {'status': 'skipped', 'reason': 'single_class_train', 'n_train': len(y_train)}

    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    metric_rows = []
    prediction_frames = []

    for split_name, eval_df in [('valid', valid_df), ('test', test_df)]:
        X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col, task)
        if len(y_eval) < config['min_valid_or_test_rows']:
            continue

        y_pred = fitted_model.predict(X_eval)

        if task == 'binary':
            y_score = get_positive_proba(fitted_model, X_eval)
            if y_score is None:
                y_score = y_pred
            metrics = binary_metrics(y_eval, y_pred, y_score)
        elif task == 'regression':
            y_score = y_pred
            metrics = regression_metrics(y_eval, y_pred)
        elif task == 'multiclass':
            y_score = y_pred
            metrics = multiclass_metrics(y_eval, y_pred)
        else:
            raise ValueError(f'Unknown task: {task}')

        row = {
            'dataset_name': DATASET_CONFIG.get('dataset_name', ''),
            'target_name': target_name,
            'target_column': target_col,
            'task': task,
            'feature_set': feature_set_name,
            'n_features': len(feature_cols),
            'model': model_name,
            'split': split_name,
            'n_train_after_target_filter': len(y_train),
            'n_eval_after_target_filter': len(y_eval),
        }
        row.update(metrics)
        metric_rows.append(row)

        prediction_frames.append(
            build_prediction_frame(
                eval_df,
                y_eval,
                y_pred,
                y_score,
                target_name,
                task,
                feature_set_name,
                model_name,
                split_name,
            )
        )

    return metric_rows, prediction_frames, {'status': 'completed', 'n_train': len(y_train)}


def run_integrated_experiments(frames, feature_sets, target_configs=TARGET_CONFIGS, config=EXPERIMENT_CONFIG):
    all_metric_rows = []
    all_prediction_frames = []
    skipped_rows = []

    all_tmp = pd.concat(frames.values(), ignore_index=True, sort=False)
    available_targets = available_target_configs(all_tmp, target_configs)

    for target_name, target_cfg in available_targets.items():
        if not should_run_name(target_name, config['target_names']):
            continue

        models = get_models_for_task(target_cfg['task'])

        for feature_set_name, feature_cols in feature_sets.items():
            if not should_run_name(feature_set_name, config['feature_set_names']):
                continue
            if len(feature_cols) == 0:
                continue

            for model_name, model in models.items():
                if not should_run_name(model_name, config['model_names']):
                    continue

                print(f'Running: target={target_name} | features={feature_set_name} | model={model_name}')

                try:
                    metric_rows, prediction_frames, status = fit_predict_single_experiment(
                        frames,
                        target_name,
                        target_cfg,
                        feature_set_name,
                        feature_cols,
                        model_name,
                        model,
                        config,
                    )
                    all_metric_rows.extend(metric_rows)
                    all_prediction_frames.extend(prediction_frames)

                    if status['status'] != 'completed':
                        skipped_rows.append({'target_name': target_name, 'feature_set': feature_set_name, 'model': model_name, **status})

                except Exception as e:
                    skipped_rows.append({'target_name': target_name, 'feature_set': feature_set_name, 'model': model_name, 'status': 'error', 'reason': str(e)})
                    print('  -> skipped/error:', e)

    metrics_df = pd.DataFrame(all_metric_rows)
    predictions_df = pd.concat(all_prediction_frames, ignore_index=True, sort=False) if all_prediction_frames else pd.DataFrame()

    if len(predictions_df) > 0:
        predictions_df = add_signal_interface(predictions_df, target_configs=target_configs)

    skipped_df = pd.DataFrame(skipped_rows)

    register_table('model_metric_summary', metrics_df)
    register_table('skipped_or_failed_experiments', skipped_df)

    if config['save_row_level_predictions'] and len(predictions_df) > 0:
        pred_path = PREDICTION_DIR / f'integrated_predictions_{RUN_TIMESTAMP}.parquet'
        predictions_df.to_parquet(pred_path, index=False)
        print('Prediction parquet exported:', pred_path)

    return metrics_df, predictions_df, skipped_df

# metrics_df, predictions_df, skipped_df = run_integrated_experiments(frames, FEATURE_SET_CONFIGS)


### Modification note — main experiment runner

This cell was updated to apply `label__rl.trainable == 1` filtering for RL-derived targets, preserve `IndexReference` in row-level predictions, and add the common `signal_score`, `direction`, and `confidence` interface after prediction generation. These changes make the prediction output suitable for downstream simulation without changing the core modelling loop.

## 10.1 Simulator-compatible prediction submission export

This section converts row-level predictions into the structured project submission format. It creates official JSON files and optional JSONL mirrors containing `metadata`, `prediction_results`, and `model_metrics`. Each prediction row includes `trade.long`, `trade.short`, `trade.no_trade`, and `signal.score` heads.

In [ ]:
# ============================================================
# Simulator-compatible prediction export
# ============================================================
# This section exports predictions in the structured format expected by
# downstream metric engines and trade simulation workflows.


def to_python_scalar(x):
    if pd.isna(x):
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    return x


def get_timeframe_value(row):
    for col in ['attr__daily_timeframe', 'attr__timeframe', 'attr__report_timeframe']:
        if col in row and pd.notna(row[col]):
            return str(row[col])
    return 'd'


def build_prediction_result_record(row, include_actual=False):
    if INDEX_COL not in row or pd.isna(row[INDEX_COL]):
        raise ValueError(f'{INDEX_COL} is required for simulator-compatible prediction export.')

    signal_score = float(np.clip(row.get(SIGNAL_SCORE_COL, row.get(SCORE_COL, 0.0)), 0.0, 1.0))
    confidence = float(np.clip(row.get(CONFIDENCE_COL, 0.5), 0.0, 1.0))
    direction = str(row.get(DIRECTION_COL, 'long'))

    # Default long-side benchmark interface.
    trade_long = signal_score
    trade_short = 0.0
    trade_no_trade = 1.0 - max(trade_long, trade_short)

    if direction == 'short':
        trade_short = signal_score
        trade_long = 0.0
        trade_no_trade = 1.0 - max(trade_long, trade_short)

    long_payload = {'label': 'trade.long', 'prediction': trade_long, 'confidence': confidence}
    short_payload = {'label': 'trade.short', 'prediction': trade_short, 'confidence': confidence}
    no_trade_payload = {'label': 'trade.no_trade', 'prediction': trade_no_trade, 'confidence': confidence}

    if include_actual:
        actual = to_python_scalar(row.get(TRUE_COL))
        long_payload['actual_value'] = actual

    timestamp_value = row.get(DATE_COL)
    if isinstance(timestamp_value, pd.Timestamp):
        timestamp_value = timestamp_value.isoformat()
    else:
        timestamp_value = str(timestamp_value)

    return {
        'index': int(row[INDEX_COL]),
        'ticker': str(row[TICKER_COL]),
        'timestamp': timestamp_value,
        'timeframe': get_timeframe_value(row),
        'predictions': {
            'trade.long': long_payload,
            'trade.short': short_payload,
            'trade.no_trade': no_trade_payload,
            'signal.score': {'label': 'signal_score', 'prediction': signal_score, 'confidence': confidence},
        },
    }


def export_prediction_submission_json(
    predictions_df,
    metrics_df,
    experiment_id,
    split,
    target_name,
    feature_set,
    model_name,
    output_root=PREDICTION_DIR,
    split_file=None,
    model_type=None,
    include_actual=False,
    export_jsonl=False,
):
    """
    Export one structured prediction file for one target-feature-model-split combination.
    The JSON file follows the required top-level structure:
    metadata, prediction_results, and model_metrics.
    """
    if predictions_df is None or len(predictions_df) == 0:
        return None
    if INDEX_COL not in predictions_df.columns:
        raise ValueError(f'{INDEX_COL} is missing from predictions_df. Re-run experiments with the updated build_prediction_frame().')

    df = predictions_df[
        (predictions_df['target_name'] == target_name)
        & (predictions_df['feature_set'] == feature_set)
        & (predictions_df['model'] == model_name)
        & (predictions_df[SPLIT_COL] == split)
    ].copy()

    if len(df) == 0:
        return None

    sort_cols = [c for c in [DATE_COL, TICKER_COL, INDEX_COL] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)

    model_id = safe_table_name(f'{model_name}_{feature_set}_{target_name}')
    if model_type is None:
        model_type = model_name

    metadata = {
        'prediction_schema_version': '1.0',
        'experiment_id': experiment_id,
        'split': split,
        'model_id': model_id,
        'model_type': model_type,
        'created_at': pd.Timestamp.utcnow().isoformat(),
        'target_name': target_name,
        'feature_set': feature_set,
        'training_dataset': DATASET_CONFIG.get('dataset_name', ''),
        'notes': 'Generated by Integrated_Experiment_Runner. signal_score is rank-normalised within target-feature-model-split; confidence is a documented distance-from-neutral proxy.',
    }

    if split_file is not None:
        metadata['split_file'] = split_file

    model_metrics = {}
    if metrics_df is not None and len(metrics_df) > 0:
        m = metrics_df[
            (metrics_df['target_name'] == target_name)
            & (metrics_df['feature_set'] == feature_set)
            & (metrics_df['model'] == model_name)
            & (metrics_df[SPLIT_COL] == split)
        ]
        if len(m) > 0:
            model_metrics = {k: to_python_scalar(v) for k, v in m.iloc[0].to_dict().items()}

    prediction_results = [build_prediction_result_record(row, include_actual=include_actual) for _, row in df.iterrows()]

    payload = {
        'metadata': metadata,
        'prediction_results': prediction_results,
        'model_metrics': model_metrics,
    }

    out_dir = Path(output_root) / safe_table_name(experiment_id).lower()
    out_dir.mkdir(parents=True, exist_ok=True)

    json_path = out_dir / f'{model_id}_{split}_predictions.json'
    with json_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    output_paths = [json_path]

    if export_jsonl:
        jsonl_path = out_dir / f'{model_id}_{split}_predictions.jsonl'
        with jsonl_path.open('w', encoding='utf-8') as f:
            f.write(json.dumps({'section': 'metadata', 'metadata': metadata}, ensure_ascii=False) + '\n')
            for rec in prediction_results:
                f.write(json.dumps({'section': 'prediction_result', 'data': rec}, ensure_ascii=False) + '\n')
            f.write(json.dumps({'section': 'model_metrics', 'model_metrics': model_metrics}, ensure_ascii=False) + '\n')
        output_paths.append(jsonl_path)

    return output_paths


def export_all_prediction_submissions(
    predictions_df,
    metrics_df,
    experiment_id=None,
    output_root=PREDICTION_DIR,
    split='test',
    include_actual=False,
    export_jsonl=False,
):
    """
    Export simulator-compatible files for all target-feature-model combinations in one split.
    """
    if experiment_id is None:
        experiment_id = DATASET_CONFIG.get('experiment_id', DATASET_CONFIG.get('dataset_name', 'experiment'))

    paths = []
    if predictions_df is None or len(predictions_df) == 0:
        return paths

    group_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    for (target_name, feature_set, model_name, split_name), _ in predictions_df.groupby(group_cols, dropna=False):
        if split_name != split:
            continue
        out = export_prediction_submission_json(
            predictions_df=predictions_df,
            metrics_df=metrics_df,
            experiment_id=experiment_id,
            split=split_name,
            target_name=target_name,
            feature_set=feature_set,
            model_name=model_name,
            output_root=output_root,
            include_actual=include_actual,
            export_jsonl=export_jsonl,
        )
        if out:
            paths.extend(out)

    submission_index = pd.DataFrame({'prediction_file': [str(p) for p in paths]})
    register_table('prediction_submission_files', submission_index)
    print(f'Exported {len(paths)} prediction submission files.')
    return paths

# Example after running experiments:
# prediction_submission_paths = export_all_prediction_submissions(
#     predictions_df=predictions_df,
#     metrics_df=metrics_df,
#     experiment_id=DATASET_CONFIG.get('dataset_name'),
#     split='test',
#     include_actual=False,
#     export_jsonl=True,
# )


### Modification note — simulator-compatible export

This new cell adds downstream trade-simulation exports. It requires `IndexReference`, `ticker`, and `timestamp` to prevent row misalignment, and it produces the `trade.long` / `trade.short` / `trade.no_trade` prediction heads requested by the project format. JSONL export is optional and can be enabled with `export_jsonl=True`.

## 11. Post-model diagnostics

This section turns row-level predictions into research-friendly diagnostic tables: daily cross-sectional Spearman, Top-K diagnostics, within-ticker ranking, subgroup diagnostics, and binary calibration tables. These outputs are essential for interpreting whether weak average performance hides useful subgroup or ranking behaviour.

In [ ]:
def run_prediction_diagnostics(predictions_df, config=EXPERIMENT_CONFIG):
    if predictions_df is None or len(predictions_df) == 0:
        print('No predictions available.')
        return {}

    diagnostic_tables = {}
    group_cols = ['target_name', 'task', 'feature_set', 'model', SPLIT_COL]

    if config['run_daily_ranking']:
        parts = []
        for keys, g in predictions_df.groupby(group_cols, dropna=False):
            one = daily_cross_sectional_spearman(g)
            if len(one) == 0:
                continue
            for col, value in zip(group_cols, keys):
                one[col] = value
            parts.append(one)
        daily_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        daily_summary = daily_df.groupby(group_cols, dropna=False).agg(
            n_days=('daily_spearman', 'count'),
            mean_daily_spearman=('daily_spearman', 'mean'),
            median_daily_spearman=('daily_spearman', 'median'),
            positive_spearman_day_rate=('daily_spearman', lambda x: (x > 0).mean()),
        ).reset_index() if len(daily_df) else pd.DataFrame()
        diagnostic_tables['daily_spearman_detail'] = register_table('daily_spearman_detail', daily_df)
        diagnostic_tables['daily_spearman_summary'] = register_table('daily_spearman_summary', daily_summary)

    if config['run_topk']:
        parts = []
        for top_pct in config['topk_percentages']:
            for keys, g in predictions_df.groupby(group_cols, dropna=False):
                one = topk_diagnostics(g, top_pct=top_pct)
                if len(one) == 0:
                    continue
                for col, value in zip(group_cols, keys):
                    one[col] = value
                parts.append(one)
        topk_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        if len(topk_df):
            agg = {
                'n_days': ('top_minus_bottom_spread', 'count'),
                'mean_top_true': ('top_mean_true', 'mean'),
                'mean_bottom_true': ('bottom_mean_true', 'mean'),
                'mean_top_minus_bottom_spread': ('top_minus_bottom_spread', 'mean'),
                'positive_spread_day_rate': ('top_minus_bottom_spread', lambda x: (x > 0).mean()),
            }
            if 'precision_at_k' in topk_df.columns:
                agg['mean_precision_at_k'] = ('precision_at_k', 'mean')
            if 'lift_at_k' in topk_df.columns:
                agg['mean_lift_at_k'] = ('lift_at_k', 'mean')
            topk_summary = topk_df.groupby(group_cols + ['top_pct'], dropna=False).agg(**agg).reset_index()
        else:
            topk_summary = pd.DataFrame()
        diagnostic_tables['topk_detail'] = register_table('topk_detail', topk_df)
        diagnostic_tables['topk_summary'] = register_table('topk_summary', topk_summary)

    if config['run_within_ticker_ranking']:
        parts = []
        for keys, g in predictions_df.groupby(group_cols, dropna=False):
            one = within_ticker_ranking(g)
            if len(one) == 0:
                continue
            for col, value in zip(group_cols, keys):
                one[col] = value
            parts.append(one)
        w_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        w_summary = w_df.groupby(group_cols, dropna=False).agg(
            n_tickers=('ticker', 'count'),
            mean_within_ticker_spearman=('within_ticker_spearman', 'mean'),
            median_within_ticker_spearman=('within_ticker_spearman', 'median'),
            positive_ticker_spearman_rate=('within_ticker_spearman', lambda x: (x > 0).mean()),
        ).reset_index() if len(w_df) else pd.DataFrame()
        diagnostic_tables['within_ticker_detail'] = register_table('within_ticker_detail', w_df)
        diagnostic_tables['within_ticker_summary'] = register_table('within_ticker_summary', w_summary)

    if config['run_subgroup_diagnostics']:
        # Sector-level diagnostics use SIC2 only. SIC code and SIC description are not used as extra group-analysis dimensions.
        for subgroup_col in [YEAR_COL, TICKER_COL, SIC2_COL]:
            if subgroup_col not in predictions_df.columns:
                continue
            parts = []
            for keys, g in predictions_df.groupby(group_cols, dropna=False):
                one = subgroup_prediction_metrics(g, subgroup_col)
                if len(one) == 0:
                    continue
                for col, value in zip(group_cols, keys):
                    one[col] = value
                parts.append(one)
            diagnostic_tables[f'subgroup_{subgroup_col}'] = register_table(
                f'subgroup_diagnostics_by_{subgroup_col}',
                pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame(),
            )

    if config['run_calibration']:
        parts = []
        binary_pred = predictions_df[predictions_df['task'] == 'binary'].copy()
        for keys, g in binary_pred.groupby(group_cols, dropna=False):
            one = calibration_table(g[TRUE_COL], g[SCORE_COL])
            if len(one) == 0:
                continue
            for col, value in zip(group_cols, keys):
                one[col] = value
            parts.append(one)
        diagnostic_tables['calibration'] = register_table(
            'binary_calibration_tables',
            pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame(),
        )

    return diagnostic_tables

# diagnostic_tables = run_prediction_diagnostics(predictions_df)


### Modification note — subgroup diagnostics

This cell now uses `SIC2` as the only sector grouping dimension. It no longer creates extra subgroup diagnostics by full SIC code or SIC description, matching the selected sector-analysis design.

## 12. Best-model comparison tables

This section creates compact tables for interpretation and dissertation writing. It identifies best models by target, best feature families, best daily-Spearman models, and best Top-K models.

In [ ]:
def build_best_model_tables(metrics_df):
    if metrics_df is None or len(metrics_df) == 0: return {}
    df = metrics_df.copy()
    def choose_main_metric(row):
        if row['task'] == 'regression': return row.get('spearman', np.nan)
        if row['task'] == 'binary': return row.get('pr_auc', row.get('roc_auc', np.nan))
        if row['task'] == 'multiclass': return row.get('macro_f1', np.nan)
        return np.nan
    df['main_metric'] = df.apply(choose_main_metric, axis=1)
    best_by_target = df.dropna(subset=['main_metric']).sort_values(['target_name', 'split', 'main_metric'], ascending=[True, True, False]).groupby(['target_name', 'split'], as_index=False).head(1).reset_index(drop=True)
    feature_family = df.dropna(subset=['main_metric']).groupby(['target_name', 'task', 'feature_set', 'split'], dropna=False).agg(best_main_metric=('main_metric', 'max'), mean_main_metric=('main_metric', 'mean'), n_models=('model', 'nunique')).reset_index().sort_values(['target_name', 'split', 'best_main_metric'], ascending=[True, True, False])
    return {'best_by_target': register_table('best_model_by_target', best_by_target), 'feature_family_comparison': register_table('feature_family_comparison', feature_family)}


def build_best_ranking_tables():
    tables = {}
    daily = EXPORTED_TABLES.get('daily_spearman_summary')
    if daily is not None and len(daily) > 0:
        best_daily = daily.dropna(subset=['mean_daily_spearman']).sort_values(['target_name', SPLIT_COL, 'mean_daily_spearman'], ascending=[True, True, False]).groupby(['target_name', SPLIT_COL], as_index=False).head(1).reset_index(drop=True)
        tables['best_daily_spearman'] = register_table('best_daily_spearman_models', best_daily)
    topk = EXPORTED_TABLES.get('topk_summary')
    if topk is not None and len(topk) > 0:
        score_col = 'mean_lift_at_k' if 'mean_lift_at_k' in topk.columns else 'mean_top_minus_bottom_spread'
        best_topk = topk.dropna(subset=[score_col]).sort_values(['target_name', SPLIT_COL, 'top_pct', score_col], ascending=[True, True, True, False]).groupby(['target_name', SPLIT_COL, 'top_pct'], as_index=False).head(1).reset_index(drop=True)
        tables['best_topk'] = register_table('best_topk_models', best_topk)
    return tables

# best_model_tables = build_best_model_tables(metrics_df)
# best_ranking_tables = build_best_ranking_tables()

## 13. Current PnL-specific temporal decile analysis

This section is separate because current PnL is a continuous trading-outcome target and benefits from temporal decile diagnostics. The purpose is to test whether higher model-predicted scores correspond to better realised current PnL over time.

In [ ]:
def predicted_decile_outcome_table(pred_df, n_deciles=10, min_rows=30):
    if pred_df is None or len(pred_df) < min_rows: return pd.DataFrame()
    df = pred_df.dropna(subset=[TRUE_COL, SCORE_COL]).copy()
    if len(df) < min_rows: return pd.DataFrame()
    df['prediction_rank_pct'] = df[SCORE_COL].rank(method='first', pct=True)
    df['prediction_decile'] = np.ceil(df['prediction_rank_pct'] * n_deciles).clip(1, n_deciles).astype(int)
    return df.groupby('prediction_decile').agg(n=(TRUE_COL, 'size'), mean_true=(TRUE_COL, 'mean'), median_true=(TRUE_COL, 'median'), std_true=(TRUE_COL, 'std'), mean_score=(SCORE_COL, 'mean'), positive_true_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') > 0).mean()), negative_true_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') < 0).mean())).reset_index()


def temporal_decile_analysis(predictions_df, target_name='rl_long_current_pnl', n_deciles=10):
    if predictions_df is None or len(predictions_df) == 0: return pd.DataFrame(), pd.DataFrame()
    df = predictions_df[predictions_df['target_name'] == target_name].copy()
    if len(df) == 0: return pd.DataFrame(), pd.DataFrame()
    group_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    detail_parts = []; year_parts = []
    for keys, g in df.groupby(group_cols, dropna=False):
        detail = predicted_decile_outcome_table(g, n_deciles=n_deciles)
        if len(detail):
            for col, value in zip(group_cols, keys): detail[col] = value
            detail_parts.append(detail)
        if YEAR_COL in g.columns:
            for year, gy in g.groupby(YEAR_COL, dropna=False):
                yd = predicted_decile_outcome_table(gy, n_deciles=n_deciles, min_rows=20)
                if len(yd):
                    for col, value in zip(group_cols, keys): yd[col] = value
                    yd[YEAR_COL] = year; year_parts.append(yd)
    detail_df = pd.concat(detail_parts, ignore_index=True, sort=False) if detail_parts else pd.DataFrame()
    year_df = pd.concat(year_parts, ignore_index=True, sort=False) if year_parts else pd.DataFrame()
    register_table('current_pnl_prediction_deciles', detail_df)
    register_table('current_pnl_prediction_deciles_by_year', year_df)
    return detail_df, year_df

# current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(predictions_df)

## 14. Optional grouped-model experiments

This section supports cases where one global panel model may be misleading because tickers or industries have very different label distributions. It trains separate models by group, such as ticker or SIC2, and exports group-level performance tables.

In [ ]:
GROUPED_MODEL_CONFIG = {
    'enabled': False,
    'group_columns': [TICKER_COL, SIC2_COL],
    'target_names': ['rl_long_current_pnl'],
    'feature_set_names': ['combined_project_b'],
    'min_train_rows_per_group': 100,
    'min_eval_rows_per_group': 30,
}


def run_grouped_model_experiments(frames, feature_sets, target_configs=TARGET_CONFIGS, grouped_config=GROUPED_MODEL_CONFIG):
    if not grouped_config.get('enabled', False):
        print('Grouped model experiments disabled.')
        return pd.DataFrame(), pd.DataFrame()

    metric_rows = []
    prediction_parts = []

    for group_col in grouped_config['group_columns']:
        if group_col not in frames['train'].columns:
            continue

        for target_name in grouped_config['target_names']:
            if target_name not in target_configs:
                continue

            target_cfg = target_configs[target_name]
            target_col = target_cfg['column']
            task = target_cfg['task']

            if target_col not in frames['train'].columns:
                continue

            train_base = filter_rows_for_target(frames['train'], target_name)
            valid_base = filter_rows_for_target(frames['valid'], target_name)
            test_base = filter_rows_for_target(frames['test'], target_name)

            models = get_models_for_task(task)

            for feature_set_name in grouped_config['feature_set_names']:
                if feature_set_name not in feature_sets:
                    continue

                feature_cols = feature_sets[feature_set_name]

                for group_value in train_base[group_col].dropna().unique():
                    train_g = train_base[train_base[group_col] == group_value].copy()
                    X_train, y_train = prepare_xy(train_g, feature_cols, target_col, task)

                    if len(y_train) < grouped_config['min_train_rows_per_group']:
                        continue
                    if task in ['binary', 'multiclass'] and y_train.nunique(dropna=True) < 2:
                        continue

                    for model_name, model in models.items():
                        if model_name.startswith('Dummy'):
                            continue

                        fitted = clone(model)
                        try:
                            fitted.fit(X_train, y_train)
                        except Exception:
                            continue

                        for split_name, eval_base in [('valid', valid_base), ('test', test_base)]:
                            eval_g = eval_base[eval_base[group_col] == group_value].copy()
                            X_eval, y_eval = prepare_xy(eval_g, feature_cols, target_col, task)

                            if len(y_eval) < grouped_config['min_eval_rows_per_group']:
                                continue

                            y_pred = fitted.predict(X_eval)

                            if task == 'binary':
                                y_score = get_positive_proba(fitted, X_eval)
                                if y_score is None:
                                    y_score = y_pred
                                metrics = binary_metrics(y_eval, y_pred, y_score)
                            elif task == 'regression':
                                y_score = y_pred
                                metrics = regression_metrics(y_eval, y_pred)
                            else:
                                y_score = y_pred
                                metrics = multiclass_metrics(y_eval, y_pred)

                            row = {
                                'group_col': group_col,
                                'group_value': group_value,
                                'target_name': target_name,
                                'task': task,
                                'feature_set': feature_set_name,
                                'model': model_name,
                                'split': split_name,
                                'n_features': len(feature_cols),
                                'n_train_after_target_filter': len(y_train),
                                'n_eval_after_target_filter': len(y_eval),
                            }
                            row.update(metrics)
                            metric_rows.append(row)

                            pred_df = build_prediction_frame(
                                eval_g,
                                y_eval,
                                y_pred,
                                y_score,
                                target_name,
                                task,
                                feature_set_name,
                                f'grouped_{group_col}_{model_name}',
                                split_name,
                            )
                            pred_df['group_model_col'] = group_col
                            pred_df['group_model_value'] = group_value
                            prediction_parts.append(pred_df)

    grouped_metrics = pd.DataFrame(metric_rows)
    grouped_predictions = pd.concat(prediction_parts, ignore_index=True, sort=False) if prediction_parts else pd.DataFrame()

    if len(grouped_predictions) > 0:
        grouped_predictions = add_signal_interface(grouped_predictions, target_configs=target_configs)

    register_table('grouped_model_metrics', grouped_metrics)

    if len(grouped_predictions) > 0:
        path = PREDICTION_DIR / f'grouped_model_predictions_{RUN_TIMESTAMP}.parquet'
        grouped_predictions.to_parquet(path, index=False)
        print('Grouped prediction parquet exported:', path)

    return grouped_metrics, grouped_predictions

# GROUPED_MODEL_CONFIG['enabled'] = True
# grouped_metrics, grouped_predictions = run_grouped_model_experiments(frames, FEATURE_SET_CONFIGS)


### Modification note — grouped models

This optional section now keeps grouped modelling limited to ticker-level and `SIC2` sector-level groups. It also applies the same RL trainable filtering and signal-interface conversion as the main experiment runner.

## 15. One-click execution block

This section provides the intended full workflow. After updating `DATASET_CONFIG`, run this cell to execute the integrated experiment pipeline. For a large dataset, start with a small subset of targets/features/models first, then expand after confirming the pipeline runs correctly.

In [ ]:
# Recommended first test run:
# EXPERIMENT_CONFIG['target_names'] = ['pi_hindsight_entry_original', 'rl_long_is_best', 'rl_long_current_pnl']
# EXPERIMENT_CONFIG['feature_set_names'] = ['combined_project_b', 'momentum_volatility_only', 'fundamentals_only']
# EXPERIMENT_CONFIG['model_names'] = ['Logistic', 'ElasticNet', 'RandomForest', 'LightGBM']

# Full workflow:
# frames, all_data = load_dataset_from_config(DATASET_CONFIG)
# frames, all_data = apply_target_construction(frames)
# label_eda_tables = run_all_label_eda(all_data)
# FEATURE_SET_CONFIGS = build_feature_sets(all_data, DATASET_CONFIG)
# metrics_df, predictions_df, skipped_df = run_integrated_experiments(frames, FEATURE_SET_CONFIGS)
# diagnostic_tables = run_prediction_diagnostics(predictions_df)
# best_model_tables = build_best_model_tables(metrics_df)
# best_ranking_tables = build_best_ranking_tables()
# current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(predictions_df)
# grouped_metrics, grouped_predictions = run_grouped_model_experiments(frames, FEATURE_SET_CONFIGS)
# prediction_submission_paths = export_all_prediction_submissions(
#     predictions_df=predictions_df,
#     metrics_df=metrics_df,
#     experiment_id=DATASET_CONFIG.get('dataset_name'),
#     split='test',
#     include_actual=False,
#     export_jsonl=True,
# )
# final_workbook_path = export_registered_tables()


## 16. Checklist

This final section creates a checklist of all tasks covered by the notebook. It is exported as part of the Excel outputs so the final results can be audited and traced back to the workflow.

In [ ]:
def build_notebook_checklist():
    rows = [
        ('1. Imports and global settings', 'Defined imports, random seed, canonical column names, output directories, IndexReference, and common signal-interface columns.', 'Keeps variables consistent across the whole notebook and preserves the downstream simulator join key.'),
        ('2. Dataset configuration', 'Dataset paths and dataset-level differences are controlled from DATASET_CONFIG.', 'Allows Universe/time horizon/pre-post changes without rewriting modelling code.'),
        ('3. Data loading', 'Created loaders for parquet/csv/json/jsonl/jsonl.zip and standardised train/valid/test frames.', 'Supports the project model-ready JSONL format while preserving IndexReference.'),
        ('4. Export manager', 'Created register_table and export_registered_tables with Excel-safe datetime handling and empty-workbook protection.', 'Ensures all EDA, metrics, diagnostics, and comparison tables can be exported reliably.'),
        ('5. Label construction', 'Added derived target columns for pi_hindsight, RL long-side labels, reward margin, and high-confidence long targets.', 'Makes label definitions explicit and reusable.'),
        ('6. Label EDA', 'Created overall, split, year, ticker, and SIC2 label EDA tables.', 'Checks sparsity, split stability, time variation, and ticker/sector heterogeneity.'),
        ('7. Feature sets', 'Created Project B, all-numeric, fundamentals, valuation, momentum/volatility, macro, and algorithmic feature sets.', 'Allows fair feature-family ablation and comparison.'),
        ('8. Modelling utilities', 'Created shared preparation/prediction functions, target-specific RL trainable filtering, and signal_score/direction/confidence conversion.', 'Reuses common code while preparing outputs for downstream trade simulation.'),
        ('9. Metrics', 'Created regression, binary, multiclass, calibration, daily ranking, Top-K, within-ticker, and subgroup metrics.', 'Adds MSE, explained variance, directional accuracy, WMAPE, MCC, Cohen kappa, log loss, and confusion-matrix cells.'),
        ('10. Main runner', 'Created integrated loop over targets, feature sets, and models with IndexReference-preserving predictions.', 'Runs most experiments from one consistent framework and exports row-level prediction parquet.'),
        ('11. Prediction submission export', 'Added official structured JSON export and optional JSONL export with trade.long, trade.short, and trade.no_trade heads.', 'Produces downstream simulator-compatible prediction files.'),
        ('12. Post-model diagnostics', 'Created daily Spearman, Top-K, within-ticker, by-year, by-ticker, by-SIC2, and calibration diagnostics.', 'Uses SIC2 as the only sector grouping dimension.'),
        ('13. Best-model tables', 'Created best-model, feature-family, best-ranking, and best-Top-K comparison tables.', 'Supports concise dissertation interpretation.'),
        ('14. Current PnL temporal deciles', 'Created current PnL prediction-decile analysis overall and by year.', 'Connects model scores to realised trading outcome quality.'),
        ('15. Optional grouped models', 'Added optional ticker/SIC2 grouped-model framework.', 'Tests whether separate group models are better than one global panel model without adding extra SIC-code group analysis.'),
        ('16. One-click execution', 'Provided a full execution block for the integrated workflow.', 'Makes the notebook easier to rerun and audit.'),
    ]
    checklist = pd.DataFrame(rows, columns=['section', 'task_completed', 'why_it_matters'])
    register_table('notebook_task_checklist', checklist)
    return checklist

notebook_task_checklist = build_notebook_checklist()
notebook_task_checklist


### Modification note — checklist update

The checklist now documents the new downstream-simulation additions, including `IndexReference`, the common signal interface, expanded metrics, and structured JSON/JSONL prediction submission exports.

## 17. Final Excel export

Run this after completing the EDA and modelling workflow. It exports all registered tables into one integrated workbook and also creates individual Excel files for each registered DataFrame.

In [ ]:
# final_workbook_path = export_registered_tables(
#     workbook_name=f'integrated_experiment_tables_{RUN_TIMESTAMP}.xlsx',
#     export_individual_files=True,
# )
# final_workbook_path